# Montage Maker

**Tap the play button below.** A new montage shows up in **My Drive / Montage Maker / Montages**.
Tap it again for another one.

What happens when you tap it:
1. Google asks to connect your Drive. Allow it.
2. It asks for your Apple ID password to pull videos from iCloud too. Press Enter to skip and use only your Drive.
3. It picks up to 8 videos at random from your Google Drive (and iCloud if you signed in), finds the best moment in each, and makes the montage on Google's free computer. Not your phone.
4. Only the finished montage is saved to your Drive.

Free.


In [ ]:
# Montage Maker: press the play button. A new montage lands in My Drive/Montage Maker/Montages.
# Run it again for another one.

import json, os, random, shutil, subprocess, sys, datetime, getpass
from pathlib import Path

MY_DRIVE = Path('/content/drive/MyDrive')
FOLDER = MY_DRIVE / 'Montage Maker'
OUT_DIR = FOLDER / 'Montages'
WORK = Path('/content/work')
ICLOUD = Path('/content/icloud')

MAX_CLIPS = 8          # most clips in one montage
CLIP_SECONDS = 3       # length of each clip
MIN_SECONDS = 1.5      # shorter videos than this are skipped; shorter than 3 s are used whole
WIDTH, HEIGHT, FPS = 1080, 1920, 30
VIDEO_TYPES = {'.mp4', '.mov', '.m4v'}
LIBRARY_START = datetime.date(2012, 1, 1)   # earliest date to look for videos
WINDOW_DAYS = 14                            # each run grabs videos from a random 2-week stretch


def run(cmd):
    r = subprocess.run([str(c) for c in cmd], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"{cmd[0]} failed:\n{r.stderr[-1500:]}")
    return r.stdout


def probe(path):
    out = run(['ffprobe', '-v', 'error', '-select_streams', 'v:0',
               '-show_entries', 'stream=width,height,color_transfer:format=duration',
               '-of', 'json', path])
    j = json.loads(out)
    s = (j.get('streams') or [{}])[0]
    return {'duration': float(j.get('format', {}).get('duration') or 0),
            'has_video': bool(s.get('width')),   # some .mp4 files are sound only
            'hdr': s.get('color_transfer') in ('smpte2084', 'arib-std-b67')}


def has_filter(name):
    return f' {name} ' in run(['ffmpeg', '-hide_banner', '-filters'])


# ---------- getting the videos ----------

def pull_from_icloud():
    """Copy videos from a random 2-week stretch of your iCloud library onto this free
    Google computer. Not your phone, not your Drive. Wiped when the session ends."""
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'icloudpd'], check=True)
    from icloudpd.cli import cli

    id_file = FOLDER / '.apple-id.txt'
    apple_id = id_file.read_text().strip() if id_file.exists() else input(
        'Apple ID email (press Enter to skip iCloud): ').strip()
    if not apple_id:
        return []
    id_file.write_text(apple_id)
    password = getpass.getpass('Apple ID password (not saved). Press Enter to skip iCloud: ')
    if not password:
        return []

    for attempt in range(6):
        span = (datetime.date.today() - LIBRARY_START).days - WINDOW_DAYS
        start = LIBRARY_START + datetime.timedelta(days=random.randint(0, max(span, 0)))
        end = start + datetime.timedelta(days=WINDOW_DAYS)
        dest = ICLOUD / start.isoformat()
        dest.mkdir(parents=True, exist_ok=True)
        print(f'Looking for videos from {start:%b %d, %Y} to {end:%b %d, %Y}...')
        sys.argv = ['icloudpd', '--username', apple_id,
                    '--password', password, '--password-provider', 'parameter',
                    '--mfa-provider', 'console',
                    '--cookie-directory', str(FOLDER / '.icloud-login'),
                    '--directory', str(dest), '--folder-structure', 'none',
                    '--skip-photos', '--skip-live-photos',
                    '--skip-created-before', start.isoformat(),
                    '--skip-created-after', end.isoformat(),
                    '--no-progress-bar', '--log-level', 'error']
        try:
            cli()
        except SystemExit:
            pass
        found = [p for p in dest.iterdir() if p.suffix.lower() in VIDEO_TYPES]
        if len(found) >= 2:
            print(f'Got {len(found)} videos.')
            return found
        print('Not enough videos there, trying another stretch.')
    return []


def videos_in_drive():
    """Every video anywhere in your Google Drive, except finished montages."""
    if not MY_DRIVE.exists():
        return []
    return [p for p in MY_DRIVE.rglob('*')
            if p.suffix.lower() in VIDEO_TYPES and OUT_DIR not in p.parents]


# ---------- picking the best moment ----------

def best_moment(video, duration):
    """Where to cut from. Measures movement and brightness 4 times a second (free, no AI
    service): more movement is better, too dark or too bright is worse. Picks one of the
    top 3 spots at random so each run can come out different."""
    stats = WORK / (video.stem + '.stats.txt')
    run(['ffmpeg', '-hide_banner', '-v', 'error', '-i', video,
         '-vf', f"fps=4,scale=160:-2,signalstats,metadata=print:file='{stats}'",
         '-an', '-f', 'null', '-'])
    frames = []   # [time, motion, brightness]
    for line in stats.read_text().splitlines():
        if line.startswith('frame:'):
            frames.append([float(line.split('pts_time:')[1]), 0.0, None])
        elif frames and 'signalstats.YDIF=' in line:
            frames[-1][1] = float(line.split('=')[1])
        elif frames and 'signalstats.YAVG=' in line:
            frames[-1][2] = float(line.split('=')[1])
    latest = max(duration - CLIP_SECONDS, 0)
    if not frames or latest == 0:
        return 0.0
    candidates = []
    for s in [x * 0.5 for x in range(int(latest / 0.5) + 1)]:
        window = [f for f in frames if s <= f[0] < s + CLIP_SECONDS and f[2] is not None]
        if not window:
            continue
        motion = sum(f[1] for f in window) / len(window)
        light = sum(f[2] for f in window) / len(window)
        penalty = 0.3 if light < 40 or light > 220 else 1.0
        candidates.append((motion * penalty, s))
    if not candidates:
        return random.uniform(0, latest)
    candidates.sort(reverse=True)
    best = candidates[0][0]
    top = []
    for score, s in candidates:   # top 3 that don't overlap and are close to the best
        if score < best * 0.5:
            break
        if all(abs(s - o) >= CLIP_SECONDS for _, o in top):
            top.append((score, s))
        if len(top) == 3:
            break
    return random.choice(top)[1]


# ---------- clip and combine ----------

def make_clip(video, start, length, hdr, out):
    color = ('zscale=t=linear:npl=100,format=gbrpf32le,zscale=p=bt709,'
             'tonemap=tonemap=hable:desat=0,zscale=t=bt709:m=bt709:r=tv,format=yuv420p,'
             if hdr and has_filter('zscale') else '')
    vf = (f'{color}scale={WIDTH}:{HEIGHT}:force_original_aspect_ratio=increase,'
          f'crop={WIDTH}:{HEIGHT},fps={FPS},setsar=1,format=yuv420p')
    run(['ffmpeg', '-y', '-hide_banner', '-v', 'error', '-ss', f'{start:.2f}', '-i', video,
         '-t', f'{length:.2f}', '-vf', vf, '-an',
         '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '18', out])
    return out


def combine(clips, out):
    inputs = []
    for c in clips:
        inputs += ['-i', c]
    joins = ''.join(f'[{i}:v]' for i in range(len(clips)))
    run(['ffmpeg', '-y', '-hide_banner', '-v', 'error', *inputs,
         '-filter_complex', f'{joins}concat=n={len(clips)}:v=1:a=0[v]', '-map', '[v]',
         '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '20', '-pix_fmt', 'yuv420p',
         '-movflags', '+faststart', out])
    want = sum(probe(c)['duration'] for c in clips)
    got = probe(out)['duration'] if out.exists() else 0
    if got < want - 0.1 * len(clips):
        raise RuntimeError(f'Montage came out short: {got:.1f}s instead of {want:.1f}s')
    return out


def make_montage(videos):
    WORK.mkdir(parents=True, exist_ok=True)
    random.shuffle(videos)
    chosen = []   # check videos in random order until there are enough good ones
    for v in videos:
        try:
            info = probe(v)
        except Exception:
            continue
        if info['has_video'] and info['duration'] >= MIN_SECONDS:
            chosen.append((v, info))
        if len(chosen) == MAX_CLIPS:
            break
    if not chosen:
        raise SystemExit('No videos long enough to clip.')
    clips = []
    for i, (v, info) in enumerate(chosen):
        print(f'Clipping {v.name}...')
        length = min(CLIP_SECONDS, info['duration'] - 0.1)
        start = best_moment(v, info['duration']) if length >= CLIP_SECONDS else 0.0
        clips.append(make_clip(v, start, length, info['hdr'], WORK / f'clip{i}.mp4'))
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    out = OUT_DIR / f"montage-{datetime.datetime.now():%Y-%m-%d-%H%M%S}.mp4"
    combine(clips, out)
    shutil.rmtree(WORK, ignore_errors=True)
    return out


# ---------- go ----------

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive is not None:
    drive.mount('/content/drive')
    FOLDER.mkdir(parents=True, exist_ok=True)
    videos = pull_from_icloud() + videos_in_drive()
    if not videos:
        raise SystemExit('No videos found in iCloud or in your Google Drive.')
    result = make_montage(videos)
    print(f'\nDone! Your montage is in My Drive/Montage Maker/Montages: {result.name}')
